## Label Distribution Summary for Flow-Level Parquet Files :: 01-12

In [1]:
import os
import pandas as pd
from collections import Counter
import pyarrow.parquet as pq

In [ ]:
folder_path = "C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets"
parquet_files = [f for f in os.listdir(folder_path) if f.endswith(".parquet")]

chunk_size = 30000

for file_name in parquet_files:
    file_path = os.path.join(folder_path, file_name)

    try:
        print(f"\n=== {file_name} ===")
        label_counter = Counter()

        parquet_file = pq.ParquetFile(file_path)
        num_rows = parquet_file.metadata.num_rows

        for i in range(0, num_rows, chunk_size):
            table = parquet_file.read_row_groups(
                range(parquet_file.num_row_groups), columns=[" Label"]
            ).slice(i, chunk_size)

            df_chunk = table.to_pandas()
            label_counter.update(df_chunk[" Label"])

        for label, count in label_counter.items():
            print(f"{label}: {count}")

    except Exception as e:
        print(f"Failed to read {file_name}: {e}")


=== DrDoS_DNS.parquet ===
DrDoS_DNS: 5071011
BENIGN: 3402

=== DrDoS_LDAP.parquet ===
DrDoS_LDAP: 2179930
BENIGN: 1612

=== DrDoS_MSSQL.parquet ===
DrDoS_MSSQL: 4522492
BENIGN: 2006

=== DrDoS_NetBIOS.parquet ===
DrDoS_NetBIOS: 4093279
BENIGN: 1707

=== DrDoS_NTP.parquet ===
DrDoS_NTP: 1202642
BENIGN: 14365

=== DrDoS_SNMP.parquet ===
DrDoS_SNMP: 5159870
BENIGN: 1507

=== DrDoS_SSDP.parquet ===
DrDoS_SSDP: 2610611
BENIGN: 763

=== DrDoS_UDP.parquet ===
DrDoS_UDP: 3134645
BENIGN: 2157

=== Syn.parquet ===
Syn: 1582289
BENIGN: 392

=== TFTP.parquet ===
TFTP: 20082580
BENIGN: 25247

=== UDPLag.parquet ===
UDP-lag: 366461
BENIGN: 3705
WebDDoS: 439


## Sampling 5,500 Attack Rows from Each Flow-Level Parquet File

In [2]:
import os
import pandas as pd
import pyarrow.parquet as pq

source_folder = "C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets"
output_folder = os.path.join(source_folder, "subset-parquets-12k")
os.makedirs(output_folder, exist_ok=True)
parquet_files = [f for f in os.listdir(source_folder) if f.endswith(".parquet")]

chunk_size = 30000
sample_target = 12000

In [3]:
for file_name in parquet_files:
    file_path = os.path.join(source_folder, file_name)
    try:
        parquet_file = pq.ParquetFile(file_path)
        total_rows = parquet_file.metadata.num_rows
        attack_rows = []

        full_labels = parquet_file.read(columns=[" Label"]).column(" Label").to_pandas()
        label_counts = full_labels.value_counts()
        main_attack = next(
            (
                label
                for label in label_counts.index
                if label not in ["BENIGN", "WebDDoS"]
            ),
            None,
        )

        if not main_attack:
            print(f"Skipping {file_name}: no valid attack label found.")
            continue
                
        for i in range(0, total_rows, chunk_size):
            table = parquet_file.read(use_threads=True).slice(i, chunk_size)
            df_chunk = table.to_pandas()
            attack_chunk = df_chunk[df_chunk[" Label"] == main_attack]

            if not attack_chunk.empty:
                attack_rows.append(attack_chunk)
                if sum(len(chunk) for chunk in attack_rows) >= sample_target:
                    break

        attack_df = pd.concat(attack_rows, ignore_index=True)
        if len(attack_df) < sample_target:
            print(f"Skipping {file_name}: not enough attack rows ({len(attack_df)})")
            continue

        sampled_attack = attack_df.sample(n=sample_target, random_state=42)
        out_path = os.path.join(output_folder, file_name)
        sampled_attack.to_parquet(out_path, index=False)
        print(f"Saved {main_attack} subset to: {out_path}")

    except Exception as e:
        print(f"Failed on {file_name}: {e}")

Saved DrDoS_DNS subset to: C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets\subset-parquets-12k\DrDoS_DNS.parquet
Saved DrDoS_LDAP subset to: C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets\subset-parquets-12k\DrDoS_LDAP.parquet
Saved DrDoS_MSSQL subset to: C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets\subset-parquets-12k\DrDoS_MSSQL.parquet
Saved DrDoS_NetBIOS subset to: C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets\subset-parquets-12k\DrDoS_NetBIOS.parquet
Saved DrDoS_NTP subset to: C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets\subset-parquets-12k\DrDoS_NTP.parquet
Saved DrDoS_SNMP subset to: C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets\subset-parquets-12k\DrDoS_SNMP.parquet


In [5]:
df = pd.read_parquet("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets/subset-parquets-12k/TFTP.parquet")

In [6]:
df.shape

(12000, 88)

In [7]:
df.head(5)

,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,SimillarHTTP,Inbound,Label
0,17116,172.16.0.5-192.168.50.1-44764-43548-6,172.16.0.5,44764,192.168.50.1,43548,6,2018-12-01 13:34:27.626635,105,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,TFTP
1,53462,172.16.0.5-192.168.50.1-48142-32593-6,172.16.0.5,48142,192.168.50.1,32593,6,2018-12-01 13:34:28.159338,0,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,TFTP
2,165111,172.16.0.5-192.168.50.1-58566-59910-6,172.16.0.5,58566,192.168.50.1,59910,6,2018-12-01 13:34:29.822393,1,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,TFTP
3,4573,172.16.0.5-192.168.50.1-54638-54638-6,172.16.0.5,54638,192.168.50.1,54638,6,2018-12-01 13:34:28.090854,6383862,4,...,0.0,109.0,109.0,6383752.0,0.0,6383752.0,6383752.0,0,1,TFTP
4,73714,172.16.0.5-192.168.50.1-44976-52493-6,172.16.0.5,44976,192.168.50.1,52493,6,2018-12-01 13:34:27.661904,1,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,TFTP


## Extracting 500 BENIGN Rows per File to Create a Balanced Benign Dataset

In [8]:
import os
import pandas as pd
import pyarrow.parquet as pq
import gc

def extract_benign_fast(
    parquet_files, 
    source_folder, 
    output_file, 
    target_per_file=1091,
    chunk_size=10000  # Larger chunks for speed
):
    """
    Fast extraction: read row groups directly, stop when we have enough.
    """
    
    all_samples = []
    
    for file_name in parquet_files:
        file_path = os.path.join(source_folder, file_name)
        print(f"Processing {file_name}...")
        
        try:
            parquet_file = pq.ParquetFile(file_path)
            samples_needed = target_per_file
            file_samples = []
            
            # Read row groups one by one (much faster than slicing)
            for row_group_idx in range(parquet_file.num_row_groups):
                if samples_needed <= 0:
                    break
                
                try:
                    # Read one row group at a time
                    row_group = parquet_file.read_row_group(row_group_idx)
                    df_chunk = row_group.to_pandas()
                    
                    # Get benign samples
                    benign_chunk = df_chunk[df_chunk[" Label"] == "BENIGN"]
                    
                    if not benign_chunk.empty:
                        # Take what we need
                        to_take = min(len(benign_chunk), samples_needed)
                        selected = benign_chunk.head(to_take).copy()
                        selected["source_file"] = file_name
                        
                        file_samples.append(selected)
                        samples_needed -= to_take
                        
                        print(f"  +{to_take} samples from row group {row_group_idx}")
                        
                        del selected, benign_chunk
                    
                    del df_chunk, row_group
                    gc.collect()
                    
                except Exception as e:
                    print(f"  Error in row group {row_group_idx}: {e}")
                    continue
            
            # Combine samples from this file
            if file_samples:
                file_df = pd.concat(file_samples, ignore_index=True)
                all_samples.append(file_df)
                print(f"  ✓ {file_name}: {len(file_df)} samples collected")
                del file_df, file_samples
            else:
                print(f"  ✗ {file_name}: No benign samples found")
            
            gc.collect()
            
        except Exception as e:
            print(f"  ✗ {file_name}: {e}")
            continue
    
    # Combine and save all at once
    if all_samples:
        print(f"\nCombining samples from {len(all_samples)} files...")
        final_df = pd.concat(all_samples, ignore_index=True)
        final_df.to_parquet(output_file, index=False)
        
        print(f"✓ Saved {len(final_df)} samples to: {output_file}")
        
        # Show breakdown
        print("\nBreakdown by file:")
        counts = final_df["source_file"].value_counts()
        for file_name, count in counts.items():
            print(f"  {file_name}: {count} samples")
        
        del final_df, all_samples
        gc.collect()
    else:
        print("No samples collected!")

In [12]:
benign_output_path = os.path.join(output_folder, "benign.parquet")
extract_benign_fast(parquet_files, source_folder, benign_output_path, target_per_file=1500)

Processing DrDoS_DNS.parquet...
  +1500 samples from row group 0
  ✓ DrDoS_DNS.parquet: 1500 samples collected
Processing DrDoS_LDAP.parquet...
  +780 samples from row group 0
  +720 samples from row group 1
  ✓ DrDoS_LDAP.parquet: 1500 samples collected
Processing DrDoS_MSSQL.parquet...
  +1014 samples from row group 0
  +96 samples from row group 1
  +390 samples from row group 2
  ✓ DrDoS_MSSQL.parquet: 1500 samples collected
Processing DrDoS_NetBIOS.parquet...
  +869 samples from row group 0
  +374 samples from row group 1
  +257 samples from row group 2
  ✓ DrDoS_NetBIOS.parquet: 1500 samples collected
Processing DrDoS_NTP.parquet...
  +1500 samples from row group 0
  ✓ DrDoS_NTP.parquet: 1500 samples collected
Processing DrDoS_SNMP.parquet...
  +823 samples from row group 0
  +352 samples from row group 1
  +136 samples from row group 2
  +76 samples from row group 3
  +113 samples from row group 4
  ✓ DrDoS_SNMP.parquet: 1500 samples collected
Processing DrDoS_SSDP.parquet...
  

In [22]:
df_benign = pd.read_parquet("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets/subset-parquets-12k/benign.parquet")

In [23]:
df_benign.shape

(14655, 88)

In [24]:
df_benign_sampled_to_12k = df_benign.sample(n=12000, random_state=42).reset_index(drop=True)
df_benign_sampled_to_12k.to_parquet(
    "C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets/subset-parquets-12k/benign.parquet")

In [25]:
df_benign_sampled_to_12k.shape

(12000, 88)

In [26]:
def compare_columns(df1, df2, name1="df1", name2="df2"):
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)

    common = cols1 & cols2
    only_in_df1 = cols1 - cols2
    only_in_df2 = cols2 - cols1

    print(f"Common columns ({len(common)}):\n{sorted(common)}\n")

    print(f"Columns only in {name1} ({len(only_in_df1)}):\n{sorted(only_in_df1)}\n")

    print(f"Columns only in {name2} ({len(only_in_df2)}):\n{sorted(only_in_df2)}\n")

    return {
        "common": list(common),
        f"only_in_{name1}": list(only_in_df1),
        f"only_in_{name2}": list(only_in_df2),
    }

compare_columns(df, df_benign, name1="df_normal", name2="df_benign_sampled_to_12k")

Common columns (88):
[' ACK Flag Count', ' Active Max', ' Active Min', ' Active Std', ' Average Packet Size', ' Avg Bwd Segment Size', ' Avg Fwd Segment Size', ' Bwd Avg Bytes/Bulk', ' Bwd Avg Packets/Bulk', ' Bwd Header Length', ' Bwd IAT Max', ' Bwd IAT Mean', ' Bwd IAT Min', ' Bwd IAT Std', ' Bwd PSH Flags', ' Bwd Packet Length Mean', ' Bwd Packet Length Min', ' Bwd Packet Length Std', ' Bwd Packets/s', ' Bwd URG Flags', ' CWE Flag Count', ' Destination IP', ' Destination Port', ' Down/Up Ratio', ' ECE Flag Count', ' Flow Duration', ' Flow IAT Max', ' Flow IAT Mean', ' Flow IAT Min', ' Flow IAT Std', ' Flow Packets/s', ' Fwd Avg Bulk Rate', ' Fwd Avg Packets/Bulk', ' Fwd Header Length', ' Fwd Header Length.1', ' Fwd IAT Max', ' Fwd IAT Mean', ' Fwd IAT Min', ' Fwd IAT Std', ' Fwd Packet Length Max', ' Fwd Packet Length Mean', ' Fwd Packet Length Min', ' Fwd Packet Length Std', ' Fwd URG Flags', ' Idle Max', ' Idle Min', ' Idle Std', ' Inbound', ' Init_Win_bytes_backward', ' Label', 

{'common': [' min_seg_size_forward',
  ' Packet Length Variance',
  ' CWE Flag Count',
  'Flow ID',
  ' Bwd IAT Mean',
  ' Bwd Packets/s',
  ' Flow IAT Min',
  ' Source Port',
  'Init_Win_bytes_forward',
  ' Flow IAT Max',
  ' Total Length of Bwd Packets',
  ' Subflow Bwd Packets',
  'Fwd Packets/s',
  ' Bwd PSH Flags',
  ' Bwd IAT Std',
  ' Active Std',
  ' Packet Length Std',
  ' Bwd Avg Packets/Bulk',
  'Fwd PSH Flags',
  ' Label',
  ' Bwd Packet Length Std',
  ' Flow IAT Mean',
  ' Flow IAT Std',
  ' Fwd IAT Max',
  ' Active Min',
  ' Bwd IAT Min',
  'Bwd Avg Bulk Rate',
  ' Init_Win_bytes_backward',
  ' Min Packet Length',
  ' Avg Fwd Segment Size',
  ' Inbound',
  ' Average Packet Size',
  'Subflow Fwd Packets',
  ' Fwd Avg Packets/Bulk',
  ' Bwd Header Length',
  ' Fwd Header Length',
  ' SYN Flag Count',
  ' Fwd IAT Std',
  ' ECE Flag Count',
  ' Total Backward Packets',
  'Fwd IAT Total',
  ' Bwd Avg Bytes/Bulk',
  ' Subflow Bwd Bytes',
  ' Max Packet Length',
  ' Subflow Fwd 

In [ ]:
print(df_benign_sampled_to_12k["source_file"].unique())

In [19]:
df_benign = df_benign.drop(columns=["source_file"])

df_benign.to_parquet(
    "C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets/subset-parquets-12k/benign.parquet",
    index=False,
)

In [29]:
df_benign_sampled_to_12k.shape

(12000, 88)

In [30]:
compare_columns(df, df_benign, name1="df_normal", name2="df_benign")

Common columns (88):
[' ACK Flag Count', ' Active Max', ' Active Min', ' Active Std', ' Average Packet Size', ' Avg Bwd Segment Size', ' Avg Fwd Segment Size', ' Bwd Avg Bytes/Bulk', ' Bwd Avg Packets/Bulk', ' Bwd Header Length', ' Bwd IAT Max', ' Bwd IAT Mean', ' Bwd IAT Min', ' Bwd IAT Std', ' Bwd PSH Flags', ' Bwd Packet Length Mean', ' Bwd Packet Length Min', ' Bwd Packet Length Std', ' Bwd Packets/s', ' Bwd URG Flags', ' CWE Flag Count', ' Destination IP', ' Destination Port', ' Down/Up Ratio', ' ECE Flag Count', ' Flow Duration', ' Flow IAT Max', ' Flow IAT Mean', ' Flow IAT Min', ' Flow IAT Std', ' Flow Packets/s', ' Fwd Avg Bulk Rate', ' Fwd Avg Packets/Bulk', ' Fwd Header Length', ' Fwd Header Length.1', ' Fwd IAT Max', ' Fwd IAT Mean', ' Fwd IAT Min', ' Fwd IAT Std', ' Fwd Packet Length Max', ' Fwd Packet Length Mean', ' Fwd Packet Length Min', ' Fwd Packet Length Std', ' Fwd URG Flags', ' Idle Max', ' Idle Min', ' Idle Std', ' Inbound', ' Init_Win_bytes_backward', ' Label', 

{'common': [' min_seg_size_forward',
  ' Packet Length Variance',
  ' CWE Flag Count',
  'Flow ID',
  ' Bwd IAT Mean',
  ' Bwd Packets/s',
  ' Flow IAT Min',
  ' Source Port',
  'Init_Win_bytes_forward',
  ' Flow IAT Max',
  ' Total Length of Bwd Packets',
  ' Subflow Bwd Packets',
  'Fwd Packets/s',
  ' Bwd PSH Flags',
  ' Bwd IAT Std',
  ' Active Std',
  ' Packet Length Std',
  ' Bwd Avg Packets/Bulk',
  'Fwd PSH Flags',
  ' Label',
  ' Bwd Packet Length Std',
  ' Flow IAT Mean',
  ' Flow IAT Std',
  ' Fwd IAT Max',
  ' Active Min',
  ' Bwd IAT Min',
  'Bwd Avg Bulk Rate',
  ' Init_Win_bytes_backward',
  ' Min Packet Length',
  ' Avg Fwd Segment Size',
  ' Inbound',
  ' Average Packet Size',
  'Subflow Fwd Packets',
  ' Fwd Avg Packets/Bulk',
  ' Bwd Header Length',
  ' Fwd Header Length',
  ' SYN Flag Count',
  ' Fwd IAT Std',
  ' ECE Flag Count',
  ' Total Backward Packets',
  'Fwd IAT Total',
  ' Bwd Avg Bytes/Bulk',
  ' Subflow Bwd Bytes',
  ' Max Packet Length',
  ' Subflow Fwd 

In [19]:
df_benign.head()

,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,SimillarHTTP,Inbound,Label
0,123,192.168.50.8_125.56.201.115_59099_80_6,192.168.50.8,59099,125.56.201.115,80,6,2018-12-01 10:51:40.379825,110861755,26,...,64223.338986,392446.0,179351.0,9.863210e+06,413171.813769,10007500.0,8632089.0,detectportal.firefox.com/success.txt,0,BENIGN
1,23,192.168.50.8_54.218.239.186_59102_443_6,192.168.50.8,59102,54.218.239.186,443,6,2018-12-01 10:51:40.504696,40335006,9,...,43.554563,90287.0,90185.0,9.993447e+06,40495.753715,10018634.0,9933709.0,0,0,BENIGN
2,126,192.168.50.253_224.0.0.5_0_0_0,192.168.50.253,0,224.0.0.5,0,0,2018-12-01 10:51:41.309691,113244633,56,...,859690.585416,2978061.0,4.0,9.188876e+06,809901.667647,9882838.0,6781893.0,0,0,BENIGN
3,91,192.168.50.8_23.15.4.11_59155_80_6,192.168.50.8,59155,23.15.4.11,80,6,2018-12-01 10:51:43.125459,95628949,21,...,177.422362,15367.0,14798.0,1.001418e+07,5184.077926,10016037.0,10000366.0,0,0,BENIGN
4,87,172.217.0.110_192.168.50.8_80_59131_6,192.168.50.8,59131,172.217.0.110,80,6,2018-12-01 10:51:43.141041,95613243,21,...,17.902514,26534.0,26473.0,1.000431e+07,21.430119,10004365.0,10004289.0,0,0,BENIGN


In [20]:
df_packet = pd.read_parquet("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/packet-level/packets/SAT-01-12-2018_0.parquet")

In [21]:
df_packet.head()

,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,length,ttl,payload_size,ip_id,...,tcp_seq,tcp_ack,tcp_urgent,icmp_type,icmp_code,flow_id,packet_position,inter_arrival_time,payload_entropy,packet_direction
0,1.543670e+09,172.16.0.5,192.168.50.1,60675.0,80.0,6,74,63,0,56352,...,6.804810e+07,0.000000e+00,0.0,NaN,NaN,172.16.0.5_192.168.50.1_60675_80_6,1,0.000000,0.0,backward
1,1.543670e+09,172.16.0.5,192.168.50.1,60675.0,80.0,6,74,63,0,56352,...,6.804810e+07,0.000000e+00,0.0,NaN,NaN,172.16.0.5_192.168.50.1_60675_80_6,2,0.000003,0.0,backward
2,1.543670e+09,192.168.50.1,172.16.0.5,80.0,60675.0,6,74,64,0,0,...,1.152121e+09,6.804810e+07,0.0,NaN,NaN,192.168.50.1_172.16.0.5_80_60675_6,1,0.000000,0.0,forward
3,1.543670e+09,192.168.50.1,172.16.0.5,80.0,60675.0,6,74,64,0,0,...,1.152121e+09,6.804810e+07,0.0,NaN,NaN,192.168.50.1_172.16.0.5_80_60675_6,2,0.000003,0.0,forward
4,1.543670e+09,172.16.0.5,192.168.50.1,60675.0,80.0,6,66,63,0,56353,...,6.804810e+07,1.152121e+09,0.0,NaN,NaN,172.16.0.5_192.168.50.1_60675_80_6,3,0.000327,0.0,backward


In [33]:
import pandas as pd
from pathlib import Path

flow_subset_dir = Path("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets/subset-parquets-12k")

flow_ids = set()
for file in flow_subset_dir.glob("*.parquet"):
    df = pd.read_parquet(file, columns=["Flow ID"])
    flow_ids.update(df["Flow ID"].astype(str).str.strip())

print(f"Total unique Flow IDs: {len(flow_ids)}")

Total unique Flow IDs: 136287


In [34]:
packet_dir = Path("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/packet-level/packets")
output_dir = Path("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/packet-level/subset-packets-12k")
output_dir.mkdir(exist_ok=True)

for packet_file in packet_dir.glob("*.parquet"):
    print(f"Processing {packet_file.name}")
    try:
        # Load with only necessary column first to check for matches
        df = pd.read_parquet(packet_file)

        if "flow_id" not in df.columns:
            print(f"'flow_id' not found in {packet_file.name}, skipping...")
            continue

        # Filter rows with matching flow IDs
        filtered = df[df["flow_id"].astype(str).str.strip().isin(flow_ids)]

        if not filtered.empty:
            output_file = output_dir / packet_file.name
            filtered.to_parquet(output_file, index=False)
            print(f"Saved {len(filtered)} packets to {output_file.name}")
        else:
            print(f"No matching packets in {packet_file.name}")
    except Exception as e:
        print(f"Error processing {packet_file.name}: {e}")

Processing SAT-01-12-2018_0.parquet
Saved 51545 packets to SAT-01-12-2018_0.parquet
Processing SAT-01-12-2018_01.parquet
Saved 283968 packets to SAT-01-12-2018_01.parquet
Processing SAT-01-12-2018_010.parquet
Saved 11883 packets to SAT-01-12-2018_010.parquet
Processing SAT-01-12-2018_0100.parquet
Saved 7604 packets to SAT-01-12-2018_0100.parquet
Processing SAT-01-12-2018_0101.parquet
Saved 5944 packets to SAT-01-12-2018_0101.parquet
Processing SAT-01-12-2018_0102.parquet
Saved 7276 packets to SAT-01-12-2018_0102.parquet
Processing SAT-01-12-2018_0103.parquet
Saved 5197 packets to SAT-01-12-2018_0103.parquet
Processing SAT-01-12-2018_0104.parquet
Saved 4302 packets to SAT-01-12-2018_0104.parquet
Processing SAT-01-12-2018_0105.parquet
Saved 7014 packets to SAT-01-12-2018_0105.parquet
Processing SAT-01-12-2018_0106.parquet
Saved 4956 packets to SAT-01-12-2018_0106.parquet
Processing SAT-01-12-2018_0107.parquet
Saved 4532 packets to SAT-01-12-2018_0107.parquet
Processing SAT-01-12-2018_010

In [35]:
import pandas as pd
from pathlib import Path

flow_subset_dir = Path(
    r"C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets/subset-parquets-12k"
)
packet_subset_dir = Path(
    r"C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/packet-level/subset-packets-12k"
)

print("Summary of packet count validation per flow file:\n")

for flow_file in flow_subset_dir.glob("*.parquet"):
    try:
        df_flow = pd.read_parquet(flow_file)
        df_flow["Total Packets"] = (
            df_flow[" Total Fwd Packets"] + df_flow[" Total Backward Packets"]
        )
        flow_ids = set(df_flow["Flow ID"].astype(str).str.strip())

        packet_counts_dict = {}

        # Loop over all packet files to count packets matching this flow file's IDs
        for packet_file in packet_subset_dir.glob("*.parquet"):
            try:
                df_packets = pd.read_parquet(packet_file, columns=["flow_id"])
                filtered_packets = df_packets[
                    df_packets["flow_id"].astype(str).str.strip().isin(flow_ids)
                ]
                counts = filtered_packets["flow_id"].value_counts()
                for flow_id, count in counts.items():
                    packet_counts_dict[flow_id] = (
                        packet_counts_dict.get(flow_id, 0) + count
                    )
            except Exception as e:
                print(f"  Error reading {packet_file.name}: {e}")

        packet_counts = pd.DataFrame(
            list(packet_counts_dict.items()), columns=["flow_id", "packet_count"]
        )

        comparison = df_flow.merge(
            packet_counts, left_on="Flow ID", right_on="flow_id", how="left"
        )
        comparison["packet_count"] = comparison["packet_count"].fillna(0).astype(int)
        comparison["diff"] = comparison["Total Packets"] - comparison["packet_count"]

        total_flows = len(comparison)
        matched_flows = (comparison["packet_count"] > 0).sum()
        mismatches = comparison[comparison["diff"] != 0]
        mismatch_count = len(mismatches)
        mismatch_pct = (mismatch_count / total_flows) * 100 if total_flows > 0 else 0

        max_diff = mismatches["diff"].abs().max() if mismatch_count > 0 else 0
        avg_diff = mismatches["diff"].abs().mean() if mismatch_count > 0 else 0

        print(f"File: {flow_file.name}")
        print(f"  Total flows: {total_flows}")
        print(f"  Flows with packets found: {matched_flows}")
        print(
            f"  Flows with mismatch in packet count: {mismatch_count} ({mismatch_pct:.2f}%)"
        )
        print(f"  Max abs difference in packet count: {max_diff}")
        print(f"  Average abs difference in packet count: {avg_diff:.2f}")
        print()

    except Exception as e:
        print(f"[❌] Failed to process {flow_file.name}: {e}")

Summary of packet count validation per flow file:

File: benign.parquet
  Total flows: 12000
  Flows with packets found: 10149
  Flows with mismatch in packet count: 11118 (92.65%)
  Max abs difference in packet count: 1859
  Average abs difference in packet count: 24.46

File: DrDoS_DNS.parquet
  Total flows: 12000
  Flows with packets found: 11677
  Flows with mismatch in packet count: 10513 (87.61%)
  Max abs difference in packet count: 2974
  Average abs difference in packet count: 107.90

File: DrDoS_LDAP.parquet
  Total flows: 12000
  Flows with packets found: 11996
  Flows with mismatch in packet count: 10394 (86.62%)
  Max abs difference in packet count: 648
  Average abs difference in packet count: 11.01

File: DrDoS_MSSQL.parquet
  Total flows: 12000
  Flows with packets found: 11974
  Flows with mismatch in packet count: 6954 (57.95%)
  Max abs difference in packet count: 61421
  Average abs difference in packet count: 20.93

File: DrDoS_NetBIOS.parquet
  Total flows: 12000


C:\Users\AGFirass\AppData\Local\Temp\ipykernel_7684\3585565029.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  comparison["packet_count"] = comparison["packet_count"].fillna(0).astype(int)


File: TFTP.parquet
  Total flows: 12000
  Flows with packets found: 0
  Flows with mismatch in packet count: 12000 (100.00%)
  Max abs difference in packet count: 21
  Average abs difference in packet count: 2.51

File: UDPLag.parquet
  Total flows: 12000
  Flows with packets found: 11996
  Flows with mismatch in packet count: 304 (2.53%)
  Max abs difference in packet count: 33
  Average abs difference in packet count: 4.25



# Concatenating the flow-level subset parquet files into a single file

In [36]:
import pandas as pd
from pathlib import Path

input_dir = Path("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/01-12/flow-level-parquets/subset-parquets-12k")

output_file = Path("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/subset/01-12/flow-level/combined_subset_12k.parquet")

parquet_files = input_dir.glob("*.parquet")
df_combined = pd.concat([pd.read_parquet(p) for p in parquet_files], ignore_index=True)
df_combined.to_parquet(output_file, index=False)

# print(f"Successfully combined {len(parquet_files)} files into {output_file}")
print(f"Successfully combined files into {output_file}")


Successfully combined files into C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\data\subset\01-12\flow-level\combined_subset_12k.parquet


In [38]:
df_subset = pd.read_parquet(output_file)
print(f"Subset DataFrame shape: {df_subset.shape}")

Subset DataFrame shape: (144000, 88)


In [39]:
print(f"Columns: {df_subset.columns.tolist()}")

Columns: ['Unnamed: 0', 'Flow ID', ' Source IP', ' Source Port', ' Destination IP', ' Destination Port', ' Protocol', ' Timestamp', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Varian

In [40]:
df_subset.columns = df_subset.columns.str.strip()
df_subset.to_parquet(output_file, index=False)
print(f"File overwritten with cleaned columns at {output_file}")

File overwritten with cleaned columns at C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\data\subset\01-12\flow-level\combined_subset_12k.parquet


In [41]:
print(f"Columns: {df_subset.columns.tolist()}")

Columns: ['Unnamed: 0', 'Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 

In [42]:
print(df_subset.dtypes)

Unnamed: 0          int64
Flow ID            object
Source IP          object
Source Port         int64
Destination IP     object
                   ...   
Idle Max          float64
Idle Min          float64
SimillarHTTP       object
Inbound             int64
Label              object
Length: 88, dtype: object


In [43]:
numerical_cols = df_subset.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df_subset.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}):\n", numerical_cols)
print(f"\nCategorical columns ({len(categorical_cols)}):\n", categorical_cols)

Numerical columns (82):
 ['Unnamed: 0', 'Source Port', 'Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'AC

In [ ]:
unique_values = df_subset["SimillarHTTP"].unique()
print("Distinct values in 'SimillarHTTP':", unique_values)

In [12]:
value_counts = df_subset["SimillarHTTP"].value_counts(dropna=False)
print(value_counts.head(20))  # top 20 most common values
print(f"Total unique values: {len(value_counts)}")

SimillarHTTP
0                                                                                                                            65477
205.174.165.72/c.php                                                                                                           139
ocsp.pki.goog/GTSGIAG3                                                                                                          49
ocsp.digicert.com/                                                                                                              27
detectportal.firefox.com/success.txt                                                                                             9
205.174.165.72null                                                                                                               7
drmokhberi.ca/favicon.ico                                                                                                        7
ocsp.comodoca.com/                                                    

In [16]:
if "SimillarHTTP" in df_subset.columns:
    df_subset = df_subset.drop(columns=["SimillarHTTP"])

# Overwrite the parquet file
df_subset.to_parquet(output_file, index=False)

print("Dropped 'SimillarHTTP' and overwrote the parquet file.")

Dropped 'SimillarHTTP' and overwrote the parquet file.


In [19]:
if "Unnamed: 0" in df_subset.columns:
    df_subset = df_subset.drop(columns=["Unnamed: 0"])

# Overwrite the parquet file
df_subset.to_parquet(output_file, index=False)

print("Dropped 'Unnamed: 0' and overwrote the parquet file.")

Dropped 'Unnamed: 0' and overwrote the parquet file.


In [ ]:
if "Unnamed: 0" in df_subset.columns:
    df_subset = df_subset.drop(columns=["Unnamed: 0"])

# Overwrite the parquet file
df_subset.to_parquet(output_file, index=False)

print("Dropped 'Unnamed: 0' and overwrote the parquet file.")

In [20]:
if "Source IP" in df_subset.columns:
    df_subset = df_subset.drop(columns=["Source IP"])

# Overwrite the parquet file
df_subset.to_parquet(output_file, index=False)

print("Dropped 'Source IP' and overwrote the parquet file.")

Dropped 'Source IP' and overwrote the parquet file.


In [21]:
if "Destination IP" in df_subset.columns:
    df_subset = df_subset.drop(columns=["Destination IP"])

# Overwrite the parquet file
df_subset.to_parquet(output_file, index=False)

print("Dropped 'Destination IP' and overwrote the parquet file.")

Dropped 'Destination IP' and overwrote the parquet file.


In [28]:
if "Flow ID" in df_subset.columns:
    df_subset = df_subset.drop(columns=["Flow ID"])

# Overwrite the parquet file
df_subset.to_parquet(output_file, index=False)

print("Dropped 'Flow ID' and overwrote the parquet file.")

Dropped 'Flow ID' and overwrote the parquet file.


In [33]:
if "Timestamp" in df_subset.columns:
    df_subset = df_subset.drop(columns=["Timestamp"])

# Overwrite the parquet file
df_subset.to_parquet(output_file, index=False)

print("Dropped 'Timestamp' and overwrote the parquet file.")

Dropped 'Timestamp' and overwrote the parquet file.


In [34]:
df_subset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65892 entries, 0 to 65891
Data columns (total 82 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Source Port                  65892 non-null  int64  
 1   Destination Port             65892 non-null  int64  
 2   Protocol                     65892 non-null  int64  
 3   Flow Duration                65892 non-null  int64  
 4   Total Fwd Packets            65892 non-null  int64  
 5   Total Backward Packets       65892 non-null  int64  
 6   Total Length of Fwd Packets  65892 non-null  float64
 7   Total Length of Bwd Packets  65892 non-null  float64
 8   Fwd Packet Length Max        65892 non-null  float64
 9   Fwd Packet Length Min        65892 non-null  float64
 10  Fwd Packet Length Mean       65892 non-null  float64
 11  Fwd Packet Length Std        65892 non-null  float64
 12  Bwd Packet Length Max        65892 non-null  float64
 13  Bwd Packet Lengt

In [35]:
numerical_cols = df_subset.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df_subset.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}):\n", numerical_cols)
print(f"\nCategorical columns ({len(categorical_cols)}):\n", categorical_cols)

Numerical columns (81):
 ['Source Port', 'Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',

In [36]:
unique_values = df_subset["Inbound"].unique()
print("Distinct values in 'Inbound':", unique_values)

Distinct values in 'Inbound': [0 1]


In [38]:
from sklearn.preprocessing import LabelEncoder
df_subset['Label'] = LabelEncoder().fit_transform(df_subset['Label'])

In [39]:
unique_values = df_subset["Label"].unique()
print("Distinct values in 'Label':", unique_values)

Distinct values in 'Label': [ 0  1  2  3  5  4  6  7  8  9 10 11]


In [40]:
numerical_cols = df_subset.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df_subset.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical columns ({len(numerical_cols)}):\n", numerical_cols)
print(f"\nCategorical columns ({len(categorical_cols)}):\n", categorical_cols)

Numerical columns (82):
 ['Source Port', 'Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',

In [41]:
df_subset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65892 entries, 0 to 65891
Data columns (total 82 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Source Port                  65892 non-null  int64  
 1   Destination Port             65892 non-null  int64  
 2   Protocol                     65892 non-null  int64  
 3   Flow Duration                65892 non-null  int64  
 4   Total Fwd Packets            65892 non-null  int64  
 5   Total Backward Packets       65892 non-null  int64  
 6   Total Length of Fwd Packets  65892 non-null  float64
 7   Total Length of Bwd Packets  65892 non-null  float64
 8   Fwd Packet Length Max        65892 non-null  float64
 9   Fwd Packet Length Min        65892 non-null  float64
 10  Fwd Packet Length Mean       65892 non-null  float64
 11  Fwd Packet Length Std        65892 non-null  float64
 12  Bwd Packet Length Max        65892 non-null  float64
 13  Bwd Packet Lengt

In [42]:
df_subset.shape

(65892, 82)

In [1]:
import torch
print(torch.cuda.is_available())

True


In [3]:
import pandas as pd
df_subset = pd.read_parquet("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/subset/01-12/flow-level/combined_subset.parquet")

In [6]:
df_subset['Label'].unique()

array(['BENIGN', 'DrDoS_DNS', 'DrDoS_LDAP', 'DrDoS_MSSQL',
       'DrDoS_NetBIOS', 'DrDoS_NTP', 'DrDoS_SNMP', 'DrDoS_SSDP',
       'DrDoS_UDP', 'Syn', 'TFTP', 'UDP-lag'], dtype=object)

In [7]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Load the dataset
df_subset = pd.read_parquet("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/subset/01-12/flow-level/combined_subset.parquet")

label_encoder = LabelEncoder()
df_subset["Label"] = label_encoder.fit_transform(df_subset["Label"])

df_subset.to_parquet("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/subset/01-12/flow-level/combined_subset.parquet")


In [8]:
df_subset['Label'].unique()

array([ 0,  1,  2,  3,  5,  4,  6,  7,  8,  9, 10, 11])

In [10]:
import numpy as np

# 1. Find columns that contain inf or -inf anywhere
inf_cols = []
for col in df_subset.columns:
    # Check if column has any inf or -inf
    if np.isinf(df_subset[col]).any():
        inf_cols.append(col)

print("Columns with inf/-inf values:", inf_cols)

Columns with inf/-inf values: ['Flow Bytes/s', 'Flow Packets/s']


In [11]:
# 2. Filter rows where any of those columns have inf or -inf
inf_rows = df_subset[np.isinf(df_subset[inf_cols]).any(axis=1)]

In [12]:
# 3. Show those rows and their Label column values
print("Rows with inf/-inf values and their labels:")
print(inf_rows[['Label'] + inf_cols])

Rows with inf/-inf values and their labels:
       Label  Flow Bytes/s  Flow Packets/s
162        0           inf             inf
385        0           inf             inf
432        0           inf             inf
442        0           inf             inf
446        0           inf             inf
...      ...           ...             ...
65331     11           inf             inf
65554     11           inf             inf
65768     11           inf             inf
65781     11           inf             inf
65853     11           inf             inf

[1869 rows x 3 columns]


In [13]:
# Optional: Count how many inf/-inf per Label
print("\nCount of inf/-inf rows per Label:")
print(inf_rows['Label'].value_counts())


Count of inf/-inf rows per Label:
Label
10    602
9     490
5     156
2     127
8     116
6      99
1      91
3      66
0      52
11     38
4      31
7       1
Name: count, dtype: int64


In [14]:
# Get distinct values for 'Flow Bytes/s'
print("Distinct values in 'Flow Bytes/s':")
print(df_subset['Flow Bytes/s'].unique())

Distinct values in 'Flow Bytes/s':
[1.24840167e+01 1.73546522e+00 0.00000000e+00 ... 9.78022596e+03
 9.65067157e+03 1.31864400e+04]


In [15]:
# Get distinct values for 'Flow Packets/s'
print("\nDistinct values in 'Flow Packets/s':")
print(df_subset['Flow Packets/s'].unique())


Distinct values in 'Flow Packets/s':
[ 0.46905265  0.47105485  0.49450467 ... 28.10409758 27.73181486
 37.72944217]


In [16]:
import numpy as np

columns = ['Flow Bytes/s', 'Flow Packets/s']

for col in columns:
    print(f"Analyzing column: {col}")
    col_values = df_subset[col]

    pos_inf_count = np.isposinf(col_values).sum()
    neg_inf_count = np.isneginf(col_values).sum()
    nan_count = col_values.isna().sum()
    finite_count = np.isfinite(col_values).sum()  # finite = not NaN and not +/-inf

    print(f" +inf count   : {pos_inf_count}")
    print(f" -inf count   : {neg_inf_count}")
    print(f" NaN count    : {nan_count}")
    print(f" Finite count : {finite_count}")

    distinct_values = col_values.dropna().replace([np.inf, -np.inf], ['inf', '-inf']).unique()
    print(f" Distinct values (with 'inf'/'-inf' as strings): {distinct_values}\n")

Analyzing column: Flow Bytes/s
 +inf count   : 767
 -inf count   : 0
 NaN count    : 1102
 Finite count : 64023
 Distinct values (with 'inf'/'-inf' as strings): [12.4840166926818 1.735465218475485 0.0 ... 9780.225956944523
 9650.671572116586 13186.44003848403]

Analyzing column: Flow Packets/s
 +inf count   : 1869
 -inf count   : 0
 NaN count    : 0
 Finite count : 64023
 Distinct values (with 'inf'/'-inf' as strings): [0.4690526503030734 0.4710548450147745 0.494504671139691 ...
 28.104097577426792 27.73181486240398 37.72944217019752]



In [17]:
def show_label_distribution(df, label_col="Label"):
    label_count = df[label_col].value_counts()
    print("Label Distribution:\n")
    print(label_count)
    return label_count

show_label_distribution(df_subset)

Label Distribution:

Label
1     5500
2     5500
3     5500
5     5500
8     5500
4     5500
6     5500
7     5500
10    5500
9     5500
11    5500
0     5392
Name: count, dtype: int64


Label
1     5500
2     5500
3     5500
5     5500
8     5500
4     5500
6     5500
7     5500
10    5500
9     5500
11    5500
0     5392
Name: count, dtype: int64

In [1]:
import pandas as pd
df_subset = pd.read_parquet("C:/Users/AGFirass/Documents/GitHub/Transformer-Based-DDoS-Detection/data/subset/01-12/flow-level/combined_subset.parquet")

In [3]:
df_subset['Protocol'].value_counts().to_dict()

{17: 50633, 6: 15081, 0: 178}